# AMB Training v3 - Fixed

Simple, robust training without aggressive filtering

In [ ]:
!pip install nbtlib -q
print("[1] ✓")

In [ ]:
import os, time, random
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import nbtlib

DEVICE = torch.device('cuda')
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("[2] ✓")

In [ ]:
# ========== CONFIG ==========
DATASET = '/kaggle/input/mc-data'
MAX_SIZE = 16
MAX_STRUCTURES = 300     # More structures
EPOCHS = 150             # More epochs
BATCH = 64
LR = 1e-3                # Higher LR with OneCycle
D_MODEL = 128
MIN_BLOCKS = 10          # Lower threshold
# ============================
print(f"[3] Config: {MAX_STRUCTURES} structs, {EPOCHS} epochs ✓")

In [ ]:
# Simple roles - don't overcomplicate
# 0=AIR, 1=SOLID (wall/floor/roof), 2=WINDOW, 3=DOOR
# Simpler = easier to learn

GLASS_IDS = {20, 102, 160, 95}  # Glass blocks
DOOR_IDS = {64, 71, 193, 194, 195, 196, 197}  # Door blocks
AIR_IDS = {0}  # Air

def simplify(blocks, sz):
    """Simplify to 4 roles: AIR=0, SOLID=1, WINDOW=2, DOOR=3"""
    out = np.zeros((sz, sz, sz), dtype=np.int64)
    w, h, d = min(blocks.shape[0], sz), min(blocks.shape[1], sz), min(blocks.shape[2], sz)
    b = blocks[:w, :h, :d]
    
    # Default: all non-air is SOLID (1)
    r = np.where(b > 0, 1, 0).astype(np.int64)
    
    # Mark windows
    for gid in GLASS_IDS:
        r[b == gid] = 2
    
    # Mark doors
    for did in DOOR_IDS:
        r[b == did] = 3
    
    out[:w, :h, :d] = r
    return out

print("[4] Simplifier: 4 roles (AIR, SOLID, WINDOW, DOOR) ✓")

In [ ]:
def load_sch(path):
    try:
        nbt = nbtlib.load(path)
        root = nbt.get('Schematic', nbt.get('', nbt))
        w, h, d = int(root['Width']), int(root['Height']), int(root['Length'])
        blocks = np.array(root.get('Blocks', root.get('BlockData', [])), dtype=np.uint8)
        if len(blocks) != w * h * d:
            return None
        return blocks.reshape(h, d, w).transpose(2, 0, 1)
    except:
        return None

# Count valid files
files = list(Path(DATASET).rglob('*.schematic'))
valid = 0
for f in files[:100]:
    b = load_sch(str(f))
    if b is not None and np.count_nonzero(b) >= MIN_BLOCKS:
        valid += 1
print(f"[5] Loader test: {valid}/100 valid ✓")

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, path, max_size, max_struct, min_blocks):
        self.states = []
        self.targets = []  # (x, y, z, block)
        self.progs = []
        
        files = list(Path(path).rglob('*.schematic'))
        random.shuffle(files)  # Randomize
        print(f"Found {len(files)} files")
        
        loaded = 0
        role_counts = {}
        
        for f in files:
            if loaded >= max_struct:
                break
            
            blocks = load_sch(str(f))
            if blocks is None:
                continue
            
            roles = simplify(blocks, max_size)
            n_blocks = np.count_nonzero(roles)
            
            if n_blocks < min_blocks:
                continue
            
            # Get positions sorted by Y (bottom-up)
            positions = []
            for x in range(max_size):
                for y in range(max_size):
                    for z in range(max_size):
                        if roles[x, y, z] > 0:
                            positions.append((x, y, z, int(roles[x, y, z])))
                            role_counts[roles[x, y, z]] = role_counts.get(roles[x, y, z], 0) + 1
            
            positions.sort(key=lambda p: (p[1], p[0], p[2]))
            
            # Generate samples
            state = np.zeros((max_size, max_size, max_size), dtype=np.int64)
            n = len(positions)
            
            for t, (x, y, z, block) in enumerate(positions):
                prog = t / max(n, 1)
                self.states.append(torch.from_numpy(state.copy()))
                self.targets.append((x, y, z, block))
                self.progs.append(prog)
                state[x, y, z] = block
            
            # STOP sample
            self.states.append(torch.from_numpy(state.copy()))
            self.targets.append((0, 0, 0, 0))  # STOP = block 0
            self.progs.append(1.0)
            
            loaded += 1
            if loaded % 50 == 0:
                print(f"  Loaded {loaded} structures, {len(self.states)} samples")
        
        print(f"\nDone: {loaded} structures, {len(self.states)} samples")
        print(f"Role distribution: {role_counts}")
    
    def __len__(self):
        return len(self.states)
    
    def __getitem__(self, i):
        x, y, z, block = self.targets[i]
        return {
            'state': self.states[i],
            'progress': torch.tensor(self.progs[i], dtype=torch.float32),
            'x': torch.tensor(x, dtype=torch.long),
            'y': torch.tensor(y, dtype=torch.long),
            'z': torch.tensor(z, dtype=torch.long),
            'block': torch.tensor(block, dtype=torch.long)
        }

print("[6] Dataset class ✓")

In [ ]:
# Simpler model - no phase conditioning (removed complexity)
class SimpleModel(nn.Module):
    def __init__(self, sz=16, d=128, num_blocks=4):  # 4 block types now
        super().__init__()
        self.sz = sz
        self.num_blocks = num_blocks
        
        self.embed = nn.Embedding(num_blocks, 16)
        self.conv = nn.Sequential(
            nn.Conv3d(16, 32, 3, padding=1, stride=2),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.Conv3d(32, 64, 3, padding=1, stride=2),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.Conv3d(64, d, 3, padding=1, stride=2),
            nn.BatchNorm3d(d),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1)
        )
        
        # Progress conditioning
        self.prog_fc = nn.Linear(1, d)
        
        # Combined feature processing
        self.fc = nn.Sequential(
            nn.Linear(d * 2, d),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        
        # Output heads
        self.pos_head = nn.Linear(d, sz ** 3)
        self.blk_head = nn.Linear(d, num_blocks)
    
    def forward(self, state, progress):
        # Encode state
        x = self.embed(state).permute(0, 4, 1, 2, 3).float()
        x = self.conv(x).flatten(1)
        
        # Progress conditioning
        p = self.prog_fc(progress.unsqueeze(-1))
        
        # Combine
        x = self.fc(torch.cat([x, p], dim=-1))
        
        return self.pos_head(x), self.blk_head(x)

m = SimpleModel(MAX_SIZE, D_MODEL, 4)
print(f"[7] Model: {sum(p.numel() for p in m.parameters()):,} params ✓")

In [ ]:
# Load dataset
print("Loading dataset...")
ds = SimpleDataset(DATASET, MAX_SIZE, MAX_STRUCTURES, MIN_BLOCKS)
print(f"[8] Loaded {len(ds)} samples ✓")

In [ ]:
# Training
if len(ds) > 0:
    loader = DataLoader(ds, BATCH, shuffle=True, num_workers=4, pin_memory=True)
    model = SimpleModel(MAX_SIZE, D_MODEL, 4).to(DEVICE)
    
    opt = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(loader))
    
    pos_fn = nn.CrossEntropyLoss()
    
    # Weight STOP token higher, others balanced
    blk_weights = torch.tensor([10.0, 1.0, 2.0, 2.0], device=DEVICE)
    blk_fn = nn.CrossEntropyLoss(weight=blk_weights)
    
    print(f"Batches: {len(loader)}, Device: {DEVICE}")
    print("=" * 70)
    
    best_loss = float('inf')
    best_pacc = 0
    
    for ep in range(EPOCHS):
        model.train()
        t0 = time.time()
        tot_loss, tot_pacc, tot_bacc, n = 0, 0, 0, 0
        
        for batch in loader:
            state = batch['state'].to(DEVICE)
            prog = batch['progress'].to(DEVICE)
            tx, ty, tz = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['z'].to(DEVICE)
            tblk = batch['block'].to(DEVICE)
            
            opt.zero_grad()
            pos_log, blk_log = model(state, prog)
            
            tgt_idx = tx * MAX_SIZE**2 + ty * MAX_SIZE + tz
            loss = pos_fn(pos_log, tgt_idx) + blk_fn(blk_log, tblk)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            scheduler.step()
            
            tot_loss += loss.item()
            tot_pacc += (pos_log.argmax(-1) == tgt_idx).float().mean().item()
            tot_bacc += (blk_log.argmax(-1) == tblk).float().mean().item()
            n += 1
        
        loss = tot_loss / n
        pacc = tot_pacc / n
        bacc = tot_bacc / n
        t = time.time() - t0
        
        if (ep + 1) % 10 == 0 or ep < 3:
            lr = scheduler.get_last_lr()[0]
            print(f"Ep {ep+1:3d}/{EPOCHS} | Loss {loss:.3f} | Pos {pacc:.3f} | Blk {bacc:.3f} | LR {lr:.2e} | {t:.1f}s")
        
        if pacc > best_pacc:
            best_pacc = pacc
            best_loss = loss
            torch.save(model.state_dict(), 'best.pt')
    
    print("=" * 70)
    print(f"[9] Done! Best: Loss={best_loss:.4f}, PosAcc={best_pacc:.3f} ✓")
else:
    print("[9] No data ✗")

In [ ]:
# Test generation
model.eval()
state = torch.zeros(1, MAX_SIZE, MAX_SIZE, MAX_SIZE, dtype=torch.long, device=DEVICE)

ROLE_NAMES = ['STOP', 'SOLID', 'WINDOW', 'DOOR']
block_counts = {}
placed = 0

for step in range(500):
    prog = torch.tensor([step / 500], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        pos_log, blk_log = model(state, prog)
    
    pos_idx = pos_log.argmax().item()
    blk = blk_log.argmax().item()
    
    if blk == 0:  # STOP
        print(f"STOP at step {step}")
        break
    
    z = pos_idx % MAX_SIZE
    y = (pos_idx // MAX_SIZE) % MAX_SIZE
    x = pos_idx // (MAX_SIZE ** 2)
    
    if state[0, x, y, z] == 0:
        state[0, x, y, z] = blk
        placed += 1
        block_counts[blk] = block_counts.get(blk, 0) + 1

print(f"\n[10] Generated {placed} blocks ✓")
for b, c in sorted(block_counts.items()):
    print(f"  {ROLE_NAMES[b]}: {c}")